In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash") 

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [4]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state


def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [5]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [6]:
initial_state = {'title': 'Rise of AI in Bangladesh'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in Bangladesh', 'outline': 'This is a detailed outline for a blog post on the "Rise of AI in Bangladesh." It aims to be engaging, informative, and structured for a general audience with an interest in technology and Bangladesh\'s development.\n\n---\n\n## Blog Title Options:\n\n*   Bangladesh\'s AI Awakening: A New Digital Frontier\n*   From Code to Progress: The Rise of AI in Bangladesh\n*   The Silicon Delta: How AI is Reshaping Bangladesh\n*   Beyond Digital Bangladesh: Charting the Course of AI Adoption\n*   AI in Bangladesh: Opportunities, Challenges, and the Road Ahead\n\n---\n\n## I. Introduction (Approx. 150-200 words)\n\n*   **A. Catchy Hook:** Start with a broad statement about the global AI revolution and its transformative power.\n    *   *Example:* "Artificial Intelligence isn\'t just a buzzword; it\'s a seismic shift reshaping industries, economies, and daily lives worldwide. But what about in the heart of South Asia, in the bustling nation of Bangla